# 29 · Plano anual: em que mês de assinatura o engajamento cai

Pergunta: quem assina o plano anual começa a desengajar a partir de qual mês?

Método: uma foto no último dia de cada mês, de fev a set/2026 (os treinos começam em 01/01/2026 e
o nível "possível churn" precisa de 45 dias de histórico). Em cada foto, cada assinatura ativa
recebe o nível de `src/engajamento.py` e o mês de casa (mês 1 = dias 0 a 29 desde o início,
mês 2 = dias 30 a 60...). As fotos são empilhadas e lidas por mês de casa.

Três fatores: assinaturas ativas em cada mês de casa, níveis de engajamento e saída (desligamento
da renovação por mês de casa e renovação no aniversário).

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento, NIVEIS

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")

assin = pd.read_parquet(P / "assinaturas_limpa.parquet")
resumo = pd.read_parquet(P / "assinaturas_resumo_eventos.parquet", columns=["user_id", "primeiro_desligamento", "ultimo_desligamento", "n_renovacao"])
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["user_id", "plano"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data"])
a = assin.merge(resumo, on="user_id", how="left", validate="one_to_one")
print(len(assin), "assinaturas -> depois da junção:", len(a))
print(a.plano_atual.value_counts().to_dict())
tp = painel.groupby("user_id").plano.nunique()
print("assinaturas com mais de um plano no painel mensal (troca de plano):", int((tp > 1).sum()))

32583 assinaturas -> depois da junção: 32583
{'mensal': 19556, 'anual': 13027}
assinaturas com mais de um plano no painel mensal (troca de plano): 0


## 1. Fotos mensais: nível de engajamento por mês de casa

In [2]:
fotos = []
for ref in pd.date_range("2026-02-28", "2026-09-30", freq="M"):
    v = a[(a.inicio <= ref) & (a.data_fim.isna() | (a.data_fim > ref))]
    e = niveis_engajamento(ativ, v.user_id, ref, inicio=v.set_index("user_id").inicio)
    e = e.merge(v[["user_id", "plano_atual", "inicio", "inicio_antes_do_log"]], on="user_id", validate="one_to_one")
    e["ref"] = ref
    e["mes_de_casa"] = ((ref - e.inicio).dt.days // 30.4375).astype(int) + 1
    print(ref.date(), "ativas:", len(v), "| anual:", int((v.plano_atual == "anual").sum()), "| mensal:", int((v.plano_atual == "mensal").sum()))
    fotos.append(e)
f = pd.concat(fotos, ignore_index=True)
f["alto"] = f.nivel == NIVEIS[0]
f["parado_30d"] = f.nivel.isin(NIVEIS[2:])
print("linhas (assinatura × foto):", len(f), "| assinaturas distintas:", f.user_id.nunique())

C:\Users\Inteli\AppData\Local\Temp\ipykernel_10184\2442861547.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  for ref in pd.date_range("2026-02-28", "2026-09-30", freq="M"):


2026-02-28 ativas: 14111 | anual: 7628 | mensal: 6483
2026-03-31 ativas: 14237 | anual: 7807 | mensal: 6430


2026-04-30 ativas: 14119 | anual: 7970 | mensal: 6149


2026-05-31 ativas: 14102 | anual: 8114 | mensal: 5988


2026-06-30 ativas: 14059 | anual: 8212 | mensal: 5847


2026-07-31 ativas: 14032 | anual: 8286 | mensal: 5746


2026-08-31 ativas: 14319 | anual: 8369 | mensal: 5950


2026-09-30 ativas: 14753 | anual: 8478 | mensal: 6275
linhas (assinatura × foto): 113732 | assinaturas distintas: 21284


In [3]:
def por_mes(d, max_mes=30):
    d = d[d.mes_de_casa <= max_mes]
    n = pd.crosstab(d.mes_de_casa, d.nivel).reindex(columns=NIVEIS, fill_value=0)
    p = n.div(n.sum(axis=1), axis=0) * 100
    p.insert(0, "assinatura-foto", n.sum(axis=1))
    p["treinos_30d (média)"] = d.groupby("mes_de_casa").treinos_30d.mean()
    return p

an = f[f.plano_atual == "anual"]
me = f[f.plano_atual == "mensal"]
print("== ANUAL: % em cada nível, por mês de casa"); t_an = por_mes(an); print(t_an)
print(); print("== MENSAL (comparação): % em cada nível, por mês de casa"); t_me = por_mes(me); print(t_me)
t_an.to_csv(S / "anual_engajamento_por_mes_de_casa.csv", encoding="utf-8-sig")

== ANUAL: % em cada nível, por mês de casa
nivel        assinatura-foto  1. alto  2. médio  3. baixo  4. possível churn  treinos_30d (média)
mes_de_casa                                                                                      
1                       3200     87.2      12.6       0.1                0.0                  9.1
2                       3206     80.9      17.2       1.7                0.3                 11.0
3                       3221     75.3      18.0       3.3                3.5                 10.3
4                       3172     68.8      19.8       4.3                7.1                  9.2
5                       3394     64.0      19.9       5.0               11.1                  8.1
6                       3352     57.3      22.5       6.1               14.1                  7.2
7                       3453     55.3      20.9       6.2               17.5                  6.7
8                       3261     52.9      22.2       5.7               19.

## 2. A mesma leitura acompanhando coortes (as mesmas assinaturas ao longo dos meses)

A leitura empilhada mistura gerações. Aqui cada linha é uma geração de assinaturas anuais, pelo mês de início.

In [4]:
an = an.assign(coorte=an.inicio.dt.to_period("M").astype(str))
for nome, col in [("% no nível ALTO", "alto"), ("% há 30+ dias sem treinar (baixo + possível churn)", "parado_30d")]:
    c = an[(an.inicio >= "2025-07-01") & (an.mes_de_casa <= 15)].pivot_table(index="coorte", columns="mes_de_casa", values=col, aggfunc="mean") * 100
    print("==", nome, "| anual, por mês de início (linhas) e mês de casa (colunas)"); print(c.round(0)); print()
print("assinaturas anuais por coorte (na primeira foto em que aparecem)")
print(an[an.inicio >= "2025-07-01"].groupby("coorte").user_id.nunique().to_dict())

== % no nível ALTO | anual, por mês de início (linhas) e mês de casa (colunas)
mes_de_casa   1    2    3    4    5    6    7    8    9    10   11   12   13   14   15
coorte                                                                                
2025-07      NaN  NaN  NaN  NaN  NaN  NaN 74.0 53.0 53.0 50.0 49.0 42.0 68.0 74.0 66.0
2025-08      NaN  NaN  NaN  NaN  NaN 48.0 58.0 56.0 56.0 53.0 47.0 44.0 81.0 77.0  NaN
2025-09      NaN  NaN  NaN  NaN 62.0 58.0 52.0 49.0 49.0 43.0 42.0 47.0 71.0  NaN  NaN
2025-10      NaN  NaN  NaN 73.0 62.0 59.0 56.0 55.0 51.0 46.0 48.0 47.0  NaN  NaN  NaN
2025-11      NaN  NaN 67.0 67.0 65.0 61.0 61.0 52.0 50.0 52.0 52.0  NaN  NaN  NaN  NaN
2025-12      NaN 72.0 78.0 71.0 69.0 59.0 49.0 48.0 50.0 47.0  NaN  NaN  NaN  NaN  NaN
2026-01     88.0 81.0 76.0 71.0 67.0 56.0 54.0 52.0 52.0  NaN  NaN  NaN  NaN  NaN  NaN
2026-02     85.0 83.0 80.0 72.0 64.0 55.0 57.0 56.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN
2026-03     87.0 82.0 77.0 69.0 61.0 54.0 55.0  NaN

In [5]:
# Trajetória individual: anuais iniciados em jan e fev/2026, vistos em todas as fotos até o corte
c = an[(an.inicio >= "2026-01-01") & (an.inicio < "2026-03-01")]
print("anuais iniciados em jan-fev/2026:", c.user_id.nunique())
prim = c[c.parado_30d].groupby("user_id").mes_de_casa.min()
tot = c.user_id.nunique()
x = prim.value_counts().sort_index().rename("contas").to_frame()
x["% da coorte"] = x.contas / tot * 100
x["% acumulado"] = x["% da coorte"].cumsum()
print("primeiro mês de casa em que a conta aparece há 30+ dias sem treinar"); print(x)
print("nunca ficou 30+ dias sem treinar até o corte:", tot - len(prim), f"({(tot - len(prim)) / tot:.1%})")

anuais iniciados em jan-fev/2026: 1048
primeiro mês de casa em que a conta aparece há 30+ dias sem treinar
             contas  % da coorte  % acumulado
mes_de_casa                                  
2                11          1.0          1.0
3                43          4.1          5.2
4                60          5.7         10.9
5                77          7.3         18.2
6                99          9.4         27.7
7                83          7.9         35.6
8                51          4.9         40.5
9                20          1.9         42.4
nunca ficou 30+ dias sem treinar até o corte: 604 (57.6%)


## 3. Quem está no primeiro ano contra quem já renovou

In [6]:
an = an.assign(ano_de_casa=np.where(an.mes_de_casa <= 12, "1º ano", np.where(an.mes_de_casa <= 24, "2º ano", "3º ano ou mais")))
n = pd.crosstab(an.ano_de_casa, an.nivel).reindex(columns=NIVEIS)
print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("assinatura-foto")))
print()
print("só a foto do corte (30/09/2026): anuais ativos por faixa de mês de casa")
k = an[an.ref == CORTE].assign(faixa=lambda d: pd.cut(d.mes_de_casa, [0, 1, 2, 3, 6, 9, 12, 18, 24, 100], labels=["1", "2", "3", "4-6", "7-9", "10-12", "13-18", "19-24", "25+"]))
n = pd.crosstab(k.faixa, k.nivel).reindex(columns=NIVEIS)
print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("anuais ativos")))

                1. alto  2. médio  3. baixo  4. possível churn  assinatura-foto
ano_de_casa                                                                    
1º ano             61.6      20.1       4.9               13.5            38310
2º ano             59.0      17.4       4.1               19.5            17109
3º ano ou mais     64.5      15.5       2.9               17.0             9445

só a foto do corte (30/09/2026): anuais ativos por faixa de mês de casa


       1. alto  2. médio  3. baixo  4. possível churn  anuais ativos
faixa                                                               
1         86.0      14.0       0.0                0.0            458
2         82.6      16.6       0.8                0.0            368
3         75.1      18.3       3.6                3.0            338
4-6       63.1      20.2       4.0               12.8           1037
7-9       54.8      21.3       5.2               18.7           1502
10-12     48.8      20.8       6.2               24.3           1084
13-18     68.3      16.5       3.4               11.8           1078
19-24     53.1      19.1       4.9               22.9           1300
25+       67.1      14.5       2.5               15.9           1313


## 4. Saída do anual: quando desliga a renovação e quantos renovam no aniversário

In [7]:
A = a[(a.plano_atual == "anual") & ~a.inicio_antes_do_log].copy()
print("anuais iniciados desde 07/2024:", len(A))
A["mes_deslig"] = ((A.primeiro_desligamento.dt.normalize() - A.inicio).dt.days // 30.4375 + 1)
d = A[A.primeiro_desligamento.notna() & (A.mes_deslig >= 1)]
print("com algum desligamento da renovação:", len(d), f"({len(d) / len(A):.1%})")
x = d.mes_deslig.clip(upper=13).astype(int).value_counts().sort_index().rename("desligamentos").to_frame()
x["% dos desligamentos"] = x.desligamentos / x.desligamentos.sum() * 100
x["% acumulado"] = x["% dos desligamentos"].cumsum()
x.index = [str(i) if i < 13 else "13+" for i in x.index]
print("mês de casa do primeiro desligamento"); print(x)
print()
# Aniversário: anuais que completariam 12 meses até 31/08/2026 (30 dias de folga antes do corte)
B = A[A.inicio <= "2025-08-31"]
saiu_1 = (~B.ativa_no_corte) & (B.duracao_dias <= 400)
print("anuais iniciados de 07/2024 a 08/2025:", len(B))
print("saíram antes do aniversário (até 330 dias):", int(((~B.ativa_no_corte) & (B.duracao_dias < 330)).sum()))
print("saíram no primeiro aniversário (330 a 400 dias):", int(((~B.ativa_no_corte) & B.duracao_dias.between(330, 400)).sum()), f"({((~B.ativa_no_corte) & B.duracao_dias.between(330, 400)).mean():.1%})")
print("passaram do primeiro aniversário:", int((~saiu_1).sum()), f"({(~saiu_1).mean():.1%})")
print(B[~B.ativa_no_corte].motivo_fim.value_counts().to_dict())
print()
# Engajamento 30 dias antes do aniversário × renovou ou não (aniversários de 17/03 a 31/08/2026)
R = B[(B.inicio >= "2025-03-17") & (B.inicio <= "2025-08-31")].copy()
R["aniv"] = R.inicio + pd.Timedelta(days=365)
R["renovou"] = ~((~R.ativa_no_corte) & (R.duracao_dias <= 400))
for dias, nome in [(31, "30 dias antes do aniversário"), (91, "90 dias antes do aniversário")]:
    Rk = R[R.aniv - pd.Timedelta(days=dias) >= pd.Timestamp("2026-02-14")]
    e = niveis_engajamento(ativ, Rk.user_id, Rk.set_index("user_id").aniv - pd.Timedelta(days=dias), inicio=Rk.set_index("user_id").inicio)
    e = e.merge(Rk[["user_id", "renovou"]], on="user_id")
    g = e.groupby("nivel").renovou.agg(anuais="size", renovaram="sum")
    g["% renovou"] = g.renovaram / g.anuais * 100
    g["% dos anuais"] = g.anuais / g.anuais.sum() * 100
    print(f"nível {nome} ({len(e)} anuais; renovaram {e.renovou.mean():.1%})"); print(g.reindex(NIVEIS)); print()

anuais iniciados desde 07/2024: 10460
com algum desligamento da renovação: 5354 (51.2%)
mês de casa do primeiro desligamento
     desligamentos  % dos desligamentos  % acumulado
1              314                  5.9          5.9
2              565                 10.6         16.4
3              608                 11.4         27.8
4              553                 10.3         38.1
5              570                 10.6         48.7
6              487                  9.1         57.8
7              294                  5.5         63.3
8              196                  3.7         67.0
9              220                  4.1         71.1
10             191                  3.6         74.7
11             474                  8.9         83.5
12             178                  3.3         86.9
13+            704                 13.1        100.0

anuais iniciados de 07/2024 a 08/2025: 5211
saíram antes do aniversário (até 330 dias): 47
saíram no primeiro aniversário (330 a 400

nível 30 dias antes do aniversário (1989 anuais; renovaram 49.9%)
                   anuais  renovaram  % renovou  % dos anuais
nivel                                                        
1. alto               912        723       79.3          45.9
2. médio              467        147       31.5          23.5
3. baixo              137         16       11.7           6.9
4. possível churn     473        106       22.4          23.8



nível 90 dias antes do aniversário (1255 anuais; renovaram 48.9%)
                   anuais  renovaram  % renovou  % dos anuais
nivel                                                        
1. alto               632        426       67.4          50.4
2. médio              258        104       40.3          20.6
3. baixo               81         19       23.5           6.5
4. possível churn     284         65       22.9          22.6

